In [54]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import seaborn as sns
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

In [55]:
df_train_data_clean = pd.read_csv("train_data_clean_full.csv")
df_train_data_clean = df_train_data_clean.drop(columns = ["Unnamed: 0", "case_id", "patientid"])
df_train_data_clean

,Hospital_code,Hospital_type_code,City_Code_Hospital,Hospital_region_code,Available Extra Rooms in Hospital,Department,Ward_Type,Ward_Facility_Code,Bed Grade,City_Code_Patient,Type of Admission,Severity of Illness,Visitors with Patient,Age,Admission_Deposit,Stay,Numero_registros_paciente
0,8,c,3,Z,3,radiotherapy,R,F,2.0,7.0,Emergency,Extreme,2,51-70,4911.0,Hasta 10 dias,14
1,2,c,5,Z,2,radiotherapy,S,F,2.0,7.0,Trauma,Extreme,2,51-70,5954.0,Más de 40 días,14
2,10,e,1,X,2,anesthesia,S,E,2.0,7.0,Trauma,Extreme,2,51-70,4745.0,Menos de 40 dias,14
3,26,b,2,Y,2,radiotherapy,R,D,2.0,7.0,Trauma,Extreme,2,51-70,7272.0,Más de 40 días,14
4,26,b,2,Y,2,radiotherapy,S,D,2.0,7.0,Trauma,Extreme,2,51-70,5558.0,Más de 40 días,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318320,6,a,6,X,3,radiotherapy,Q,F,4.0,23.0,Emergency,Moderate,3,21-50,4144.0,Menos de 40 dias,1
318321,24,a,1,X,2,anesthesia,Q,E,4.0,8.0,Urgent,Moderate,4,71-100,6699.0,Menos de 40 dias,1
318322,7,a,4,X,3,gynecology,R,F,4.0,10.0,Emergency,Minor,3,71-100,4235.0,Menos de 40 dias,1
318323,11,b,2,Y,3,anesthesia,Q,D,3.0,8.0,Trauma,Minor,5,0-20,3761.0,Menos de 40 dias,1


## SEPARAMOS TRAIN Y TEST

In [56]:
X = df_train_data_clean.drop(columns = "Stay")
y = df_train_data_clean["Stay"]

In [57]:
# Pdemos separar directamente en 4 tablas con el nombre propio de cada cual:
X_train, X_test, y_train, y_test,  = train_test_split(
    X, y,
    train_size = 0.75, # Separará el 75% de los datos para training, Selección aleatoria
    test_size = 0.25,   # Separará el 25% de los datos para testeo, Selección aleatoria
    random_state = 10
)

## GENERAMOS DUMMIES

In [58]:
X_train.columns

Index(['Hospital_code', 'Hospital_type_code', 'City_Code_Hospital',
       'Hospital_region_code', 'Available Extra Rooms in Hospital',
       'Department', 'Ward_Type', 'Ward_Facility_Code', 'Bed Grade',
       'City_Code_Patient', 'Type of Admission', 'Severity of Illness',
       'Visitors with Patient', 'Age', 'Admission_Deposit',
       'Numero_registros_paciente'],
      dtype='object')

In [59]:
X_train = pd.get_dummies(
    X_train,
    columns= [
        "Hospital_code",
        "Hospital_type_code",
        "City_Code_Hospital",
        "Hospital_region_code",
        "Department",
        "Ward_Type",
        'Ward_Facility_Code',
        "Bed Grade",
        "City_Code_Patient",
        "Type of Admission",
        "Severity of Illness",
        "Age",
    ],
    drop_first = True  # Me elimina una columna porque siempre sobra en el modelo. No es necesaria
)

X_test = pd.get_dummies(
    X_test,
    columns= [
        "Hospital_code",
        "Hospital_type_code",
        "City_Code_Hospital",
        "Hospital_region_code",
        "Department",
        "Ward_Type",
        'Ward_Facility_Code',
        "Bed Grade",
        "City_Code_Patient",
        "Type of Admission",
        "Severity of Illness",
        "Age",
    ],
    drop_first = True  # Me elimina una columna porque siempre sobra en el modelo. No es necesaria
)

In [60]:
# Nos aseguramos que no haya puesto más columnas en una tabla que en otra porque hubieran más categorías en alguna columna:
X_train.columns.size

114

In [61]:
X_test.columns.size

114

In [62]:
correlacion = X_train.corr()
correlacion

,Available Extra Rooms in Hospital,Visitors with Patient,Admission_Deposit,Numero_registros_paciente,Hospital_code_2,Hospital_code_3,Hospital_code_4,Hospital_code_5,Hospital_code_6,Hospital_code_7,...,City_Code_Patient_8.0,City_Code_Patient_9.0,City_Code_Patient_other,Type of Admission_Trauma,Type of Admission_Urgent,Severity of Illness_Minor,Severity of Illness_Moderate,Age_21-50,Age_51-70,Age_71-100
Available Extra Rooms in Hospital,1.000000,0.095097,-0.142714,-0.054594,-0.055650,0.022994,-0.021099,0.008492,0.050862,-0.013212,...,-0.029401,-0.002642,0.007688,-0.048660,0.017267,-0.005810,0.014144,0.190770,-0.153817,-0.059803
Visitors with Patient,0.095097,1.000000,-0.149583,0.010201,-0.006596,-0.008964,-0.022632,0.002287,0.031804,-0.019728,...,0.003199,-0.017192,0.014435,-0.024345,-0.078004,-0.037235,0.002143,-0.074556,0.082166,0.016304
Admission_Deposit,-0.142714,-0.149583,1.000000,0.036639,0.014686,-0.006506,0.035773,0.006716,-0.071400,0.041751,...,0.017871,0.004566,-0.009174,0.092837,-0.021930,0.055698,-0.006303,-0.049612,0.021092,0.029516
Numero_registros_paciente,-0.054594,0.010201,0.036639,1.000000,0.026906,0.010526,0.010485,-0.022959,0.010107,0.003300,...,0.047400,0.003446,-0.033166,0.079855,0.015544,-0.218121,0.053701,-0.031606,0.012695,0.061280
Hospital_code_2,-0.055650,-0.006596,0.014686,0.026906,1.000000,-0.019370,-0.008020,-0.016551,-0.033413,-0.008161,...,-0.027931,0.042399,-0.004947,0.007354,0.019118,-0.006056,0.006346,-0.001070,-0.000452,0.003903
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Severity of Illness_Minor,-0.005810,-0.037235,0.055698,-0.218121,-0.006056,0.002339,0.000555,0.022383,0.014736,0.004274,...,-0.060256,0.018473,0.019195,-0.061191,0.012063,1.000000,-0.673232,-0.007725,-0.057901,-0.005939
Severity of Illness_Moderate,0.014144,0.002143,-0.006303,0.053701,0.006346,-0.001258,-0.002111,-0.008636,-0.000702,-0.000251,...,0.013748,-0.003802,-0.008212,0.032905,-0.000642,-0.673232,1.000000,0.034270,0.006525,-0.016116
Age_21-50,0.190770,-0.074556,-0.049612,-0.031606,-0.001070,-0.001233,-0.003364,0.004037,0.003363,0.001384,...,-0.028162,0.006964,0.005692,-0.023453,0.038850,-0.007725,0.034270,1.000000,-0.623653,-0.428824
Age_51-70,-0.153817,0.082166,0.021092,0.012695,-0.000452,-0.003999,0.001509,-0.001059,0.000127,0.002158,...,0.017400,-0.005814,-0.000177,0.044439,-0.047208,-0.057901,0.006525,-0.623653,1.000000,-0.239259


In [63]:
# Si queremos verlo en 0 y 1:
X_train = X_train * 1
X_test = X_test * 1

In [64]:
X_test

,Available Extra Rooms in Hospital,Visitors with Patient,Admission_Deposit,Numero_registros_paciente,Hospital_code_2,Hospital_code_3,Hospital_code_4,Hospital_code_5,Hospital_code_6,Hospital_code_7,...,City_Code_Patient_8.0,City_Code_Patient_9.0,City_Code_Patient_other,Type of Admission_Trauma,Type of Admission_Urgent,Severity of Illness_Minor,Severity of Illness_Moderate,Age_21-50,Age_51-70,Age_71-100
67827,2,4,5193.0,5,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
292257,4,2,4464.0,2,0,1,0,0,0,0,...,0,0,0,0,1,0,1,1,0,0
196656,2,2,4246.0,8,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
243694,2,3,6027.0,3,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
129280,3,2,4635.0,4,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25999,2,4,6752.0,4,0,0,0,0,0,0,...,0,0,0,1,0,1,0,1,0,0
45855,4,3,4247.0,1,0,0,0,0,0,0,...,1,0,0,0,0,1,0,1,0,0
181883,3,4,4792.0,3,0,0,0,0,1,0,...,0,0,0,1,0,1,0,1,0,0
309533,3,2,4908.0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0


## GENERACIÓN DEL MODELO - RESTRINGIDO

In [45]:
for i in range(1,20):
    modelo = DecisionTreeClassifier(max_depth = i)
    modelo.fit(X_train, y_train)    
    
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(i, "->", np.round(acc,3))

1 -> 0.738
2 -> 0.738
3 -> 0.749
4 -> 0.755
5 -> 0.754
6 -> 0.763
7 -> 0.764
8 -> 0.768
9 -> 0.768
10 -> 0.767
11 -> 0.767
12 -> 0.766
13 -> 0.764
14 -> 0.763
15 -> 0.758
16 -> 0.756
17 -> 0.751
18 -> 0.747
19 -> 0.741


## RANDOM FOREST

In [65]:
from sklearn.ensemble import RandomForestClassifier

In [66]:
# Por defecto, el modelo crea 100 árboles
modelo = RandomForestClassifier(max_depth = 10)
modelo.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [67]:
y_pred = modelo.predict(X_test)
accuracy_score(y_test, y_pred)

0.7477067678620793

In [68]:
y_pred

array(['Menos de 40 dias', 'Menos de 40 dias', 'Menos de 40 dias', ...,
       'Menos de 40 dias', 'Menos de 40 dias', 'Menos de 40 dias'],
      shape=(79582,), dtype=object)

In [52]:
# Cálculo de auc
from sklearn.metrics import roc_auc_score

proba = modelo.predict_proba(X_test)  # shape = (n_samples, 3)

auc_ovr_macro = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
auc_ovr_weighted = roc_auc_score(y_test, proba, multi_class="ovr", average="weighted")

print("AUC OvR macro:", auc_ovr_macro)
print("AUC OvR weighted:", auc_ovr_weighted)

AUC OvR macro: 0.7990367313937066
AUC OvR weighted: 0.7838573868633926
